In [4]:
import json
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS, PAPILO_PRESOLVED_MODELS
from src.detection.pipstools_backend import PipstoolsDetection

# ── MACRO ─────────────────────────────────────────────────────────────────────
GENERATE_MIP = False   # True → score _mip models; False → score LP models
# ──────────────────────────────────────────────────────────────────────────────

_suffix = "_mip" if GENERATE_MIP else ""

MODEL_SCORING_OUT = Path(f"/data/energy-system-preprocessing/scoring{_suffix}/models")
MODEL_SCORING_OUT.mkdir(parents=True, exist_ok=True)
PAPILO_SCORING_OUT = Path(f"/data/energy-system-preprocessing/scoring{_suffix}/papilo")
PAPILO_SCORING_OUT.mkdir(parents=True, exist_ok=True)


def region_k(name: str) -> int | None:
    m = re.match(r"^r(\d+)_", name)
    return int(m.group(1)) if m else None


def _is_target(name: str) -> bool:
    if region_k(name) is None:
        return False
    return name.endswith("_mip") if GENERATE_MIP else not name.endswith("_mip")


def detection_params_str(detector: PipstoolsDetection, k: int) -> str:
    vd = detector._var_dense if detector._var_dense is not None else "none"
    return f"k{k}-{detector._hypergraph}-{detector._hg_objective}-vd{vd}"


def score_model(model_dir: Path, out_dir: Path, mps_name: str, k: int) -> bool:
    name = model_dir.name
    mps_in = model_dir / mps_name

    if not mps_in.exists():
        print(f"  SKIP: no {mps_name} in {model_dir}")
        return False

    detector = PipstoolsDetection(mpsreader="highs")
    params = detection_params_str(detector, k)
    out_file = out_dir / f"{name}-{params}.json"

    if out_file.exists():
        print(f"  skipped (already exists)", flush=True)
        return True
    result = detector.detect(mps_in, k=k)
    out_file.write_text(json.dumps(result.to_dict(), indent=2))
    return True

In [5]:
def r_models(root: Path) -> list[tuple[Path, int]]:
    return sorted(
        [(p, region_k(p.name)) for p in root.iterdir() if _is_target(p.name)],
        key=lambda x: x[0].name,
    )

orig_models = r_models(MODELS)
papilo_models = r_models(PAPILO_PRESOLVED_MODELS)
print(f"{len(orig_models)} original r-N models, {len(papilo_models)} papilo-presolved r-N models")

520 original r-N models, 520 papilo-presolved r-N models


In [ ]:
failed = []

print("=== Original models ===")
for i, (model_dir, k) in enumerate(orig_models, 1):
    print(f"[{i}/{len(orig_models)}] {model_dir.name}  (k={k})", flush=True)
    ok = score_model(model_dir, MODEL_SCORING_OUT, "original.mps", k)
    if not ok:
        failed.append(("original", model_dir.name))
    print(flush=True)

print("=== Papilo-presolved models ===")
for i, (model_dir, k) in enumerate(papilo_models, 1):
    print(f"[{i}/{len(papilo_models)}] {model_dir.name}  (k={k})", flush=True)

    # TODO: Fix this error rather than suppress.
    try:
        ok = score_model(model_dir, PAPILO_SCORING_OUT, "reduced.mps", k)
    except:
        print("somethings off")

    if not ok:
        failed.append(("papilo", model_dir.name))
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    total = len(orig_models) + len(papilo_models)
    print(f"\nAll {total} models scored successfully.")

=== Original models ===
[1/520] r10_res168_f0.0000_t0.0192  (k=10)
  skipped (already exists)

[2/520] r10_res168_f0.0000_t0.0192_cf0.75  (k=10)
  skipped (already exists)

[3/520] r10_res168_f0.0000_t0.0833  (k=10)
  skipped (already exists)

[4/520] r10_res168_f0.0000_t0.0833_cf0.75  (k=10)
  skipped (already exists)

[5/520] r10_res168_f0.0000_t1.0000  (k=10)
  skipped (already exists)

[6/520] r10_res168_f0.0000_t1.0000_cf0.75  (k=10)
  skipped (already exists)

[7/520] r10_res168_f0.2500_t0.5000  (k=10)
  skipped (already exists)

[8/520] r10_res168_f0.2500_t0.5000_cf0.75  (k=10)
  skipped (already exists)

[9/520] r10_res168_f0.5000_t1.0000  (k=10)
  skipped (already exists)

[10/520] r10_res168_f0.5000_t1.0000_cf0.75  (k=10)
  skipped (already exists)

[11/520] r10_res1_f0.0000_t0.0192  (k=10)
  skipped (already exists)

[12/520] r10_res1_f0.0000_t0.0192_cf0.75  (k=10)
  skipped (already exists)

[13/520] r10_res1_f0.0000_t0.0833  (k=10)
  skipped (already exists)

[14/520] r10_

[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized



[5/520] r10_res168_f0.0000_t1.0000  (k=10)
  skipped (already exists)

[6/520] r10_res168_f0.0000_t1.0000_cf0.75  (k=10)
Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Input File Format:                  hMetis (-> hypergraph)
  Preset:                             quality
  k:                                  10
  epsilon:                            0.001
  Objective:                          soed
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            50000
-------------------------------------------------------------------------------
Preproc

[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Input File Format:                  hMetis (-> hypergraph)
  Preset:                             quality
  k:                                  10
  epsilon:                            0.001
  Objective:                          soed
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            50000
-------------------------------------------------------------------------------
Preprocessing Parameters:
  Use Community Detection:            true
  Disable C. D. for Mesh Graphs:      true

  Community Dete

...

[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Input File Format:                  hMetis (-> hypergraph)
  Preset:                             quality
  k:                                  9
  epsilon:                            0.001
  Objective:                          soed
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            50000
-------------------------------------------------------------------------------
Preprocessing Parameters:
  Use Community Detection:            true
  Disable C. D. for Mesh Graphs:      true

  Community Detec

[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized


Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Input File Format:                  hMetis (-> hypergraph)
  Preset:                             quality
  k:                                  9
  epsilon:                            0.001
  Objective:                          soed
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            50000
-------------------------------------------------------------------------------
Preprocessing Parameters:
  Use Community Detection:            true
  Disable C. D. for Mesh Graphs:      true

  Community Detec

[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
[WARNING] Mt-KaHyPar is already initialized
